# RouterV3 -- Colab environment bring-up

Builds a headless bridge to KiCad's real interactive router (`PNS::ROUTER`) by compiling a small pybind11 module *inside* a KiCad 9.0.8 source checkout, then smoke-tests it by routing one net on a toy board.

**Status: unverified.** The C++ in `pcbworld/engine/cpp/` was written from reading the KiCad 9.0.8 source directly (see `docs/engine_access.md` for citations) but has never been compiled. Expect this notebook to need iteration -- missing link targets, include path issues, CMake flags that don't exist on this KiCad version, etc. Run it cell by cell and report back whatever errors show up.

Rough time budget: ~10 min installing build deps, ~30-60 min compiling KiCad + the bridge (first run only -- cache `build/` to Drive afterwards to skip this on future sessions).

## 1. System build dependencies
Uses KiCad's own PPA to pull the exact build-dependency list for 9.0, rather than hand-listing wxWidgets/Boost/OCCT/etc versions ourselves.

In [ ]:
%%bash
set -e
sudo add-apt-repository -y ppa:kicad/kicad-9.0-releases
sudo apt-get update -qq
sudo apt-get build-dep -y kicad
sudo apt-get install -y cmake ninja-build
pip install -q pybind11

## 2. Clone this repo + KiCad source
Optionally point `DRIVE_CACHE` at a Google Drive folder to persist the KiCad build directory across sessions (a full build is the slow part).

In [ ]:
import os

WORKDIR = "/content"
REPO_URL = "https://github.com/Klutzhehe/Routerv3.git"
KICAD_TAG = "9.0.8"

os.chdir(WORKDIR)

In [ ]:
%%bash -s "$REPO_URL"
set -e
cd /content
if [ ! -d Routerv3 ]; then git clone "$1" Routerv3; fi

In [ ]:
%%bash -s "$KICAD_TAG"
set -e
cd /content
if [ ! -d kicad-src ]; then
  git clone --depth 1 --branch "$1" https://gitlab.com/kicad/code/kicad.git kicad-src
fi

## 3. Wire the bridge into KiCad's build
Copies `pcbworld/engine/cpp` into the KiCad checkout and appends an `add_subdirectory()` to `pcbnew/CMakeLists.txt` so it's configured *after* the `pnsrouter`/`pcbcommon`/etc targets it links against are defined.

In [ ]:
%%bash
set -e
cd /content
rm -rf kicad-src/pcbworld_bridge
cp -r Routerv3/pcbworld/engine/cpp kicad-src/pcbworld_bridge

MARKER='add_subdirectory( ${CMAKE_SOURCE_DIR}/pcbworld_bridge'
if ! grep -qF "$MARKER" kicad-src/pcbnew/CMakeLists.txt; then
  printf '\nadd_subdirectory( ${CMAKE_SOURCE_DIR}/pcbworld_bridge ${CMAKE_BINARY_DIR}/pcbworld_bridge )\n' >> kicad-src/pcbnew/CMakeLists.txt
fi

## 4. Configure + build
Only builds the `pcbworld_pns_bridge` target and its dependencies (not the full KiCad GUI app), to keep the first build shorter. Flags below are a starting guess -- likely need adjusting once real CMake errors show up.

In [ ]:
%%bash
set -e
cd /content/kicad-src
mkdir -p build
cmake -S . -B build -G Ninja \
  -DCMAKE_BUILD_TYPE=Release \
  -DKICAD_BUILD_QA_TESTS=OFF \
  -DKICAD_SCRIPTING_WXPYTHON=OFF \
  -DKICAD_BUILD_I18N=OFF

In [ ]:
%%bash
set -e
cd /content/kicad-src
cmake --build build --target pcbworld_pns_bridge -j"$(nproc)"

## 5. Smoke test
Import the compiled module and confirm it loads.

In [ ]:
%%bash
find /content/kicad-src/build -iname 'pcbworld_pns_bridge*'

In [ ]:
import sys, glob

matches = glob.glob("/content/kicad-src/build/**/pcbworld_pns_bridge*.so", recursive=True)
assert matches, "build didn't produce the extension -- check step 4 output"
sys.path.insert(0, str(__import__("pathlib").Path(matches[0]).parent))

import pcbworld_pns_bridge as bridge

b = bridge.PNSBridge()
print(bridge.__file__)
print("module loaded OK")

## 6. Route a toy board end to end
TODO once step 5 passes: generate/load a minimal 2-pad board, `load_board`, `query_hover_items` on each pad, `start_route` -> `push` -> `fix` -> `commit_routing`, `save_board`, and open the result to confirm a real track was drawn.